# ⚔️ Day 03a: SVM — Maximum Margin & Kernel Trick

---

## 🎯 What You'll Master Today
- Maximum margin intuition
- Support vectors and decision boundary
- Kernel trick: RBF, polynomial, linear
- Soft margin (C parameter)
- sklearn SVM with hyperparameter tuning

**Goal:** Explain SVM math and choose the right kernel in an interview.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  SVM — SUPPORT VECTOR MACHINE                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Goal: Find the hyperplane with MAXIMUM MARGIN                  ║
║                                                                   ║
║       ×  ×                    ○  ○                               ║
║     ×   ×  ← margin → |   ← margin →  ○  ○                    ║
║       × [SV]  ──────── | ────────  [SV]  ○                     ║
║     ×   ×              |              ○  ○                       ║
║                    DECISION                                       ║
║                   BOUNDARY                                        ║
║                                                                   ║
║  [SV] = Support Vectors (closest points to boundary)            ║
║  margin = 2 / ||w||                                              ║
║                                                                   ║
║  Hard margin: No points allowed in margin                       ║
║  Soft margin: Some violations allowed (controlled by C)         ║
║    C large → narrow margin (overfit)                            ║
║    C small → wide margin (underfit)                             ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.datasets import make_classification, make_moons, make_circles
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

# ============================================================
# Linear SVM — Linearly Separable Data
# ============================================================

X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                           n_informative=2, random_state=42, n_clusters_per_class=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Always scale for SVM!
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Linear SVM
svm_linear = SVC(kernel='linear', C=1.0)
svm_linear.fit(X_train_s, y_train)

print(f"Linear SVM accuracy: {svm_linear.score(X_test_s, y_test):.3f}")
print(f"Support vectors: {svm_linear.n_support_}")
print(f"Total SVs: {svm_linear.support_vectors_.shape[0]}/{len(X_train)}")

In [ ]:
# ============================================================
# Kernel Trick — When Data Isn't Linearly Separable
# ============================================================

# Non-linear data
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)
X_circles, y_circles = make_circles(n_samples=300, noise=0.1, factor=0.3, random_state=42)

scaler = StandardScaler()

# Try different kernels on moons data
kernels = ['linear', 'poly', 'rbf']

for kernel in kernels:
    X_tr, X_te, y_tr, y_te = train_test_split(X_moons, y_moons, test_size=0.2, random_state=42)
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    
    svm = SVC(kernel=kernel, C=1.0)
    svm.fit(X_tr_s, y_tr)
    acc = svm.score(X_te_s, y_te)
    print(f"{kernel:8s} kernel: acc={acc:.3f}, SVs={svm.support_vectors_.shape[0]}")

print("\n💡 RBF handles non-linear boundaries. Linear fails on non-linear data.")

In [ ]:
# ============================================================
# C Parameter — Soft Margin Control
# ============================================================

X_tr, X_te, y_tr, y_te = train_test_split(X_moons, y_moons, test_size=0.2, random_state=42)
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
print("C parameter effect (RBF kernel):")
for C in C_values:
    svm = SVC(kernel='rbf', C=C)
    svm.fit(X_tr_s, y_tr)
    train_acc = svm.score(X_tr_s, y_tr)
    test_acc = svm.score(X_te_s, y_te)
    print(f"  C={C:6.2f}: train={train_acc:.3f}, test={test_acc:.3f}, SVs={svm.support_vectors_.shape[0]}")

print("\n💡 Small C → more SVs (wider margin, simpler boundary)")
print("   Large C → fewer SVs (narrower margin, complex boundary)")

In [ ]:
# ============================================================
# Kernel Details — What Each Does
# ============================================================

print("KERNEL FUNCTIONS:")
print("="*60)
print(f"Linear:     K(x,y) = xᵀy")
print(f"Polynomial: K(x,y) = (γ·xᵀy + r)^d")
print(f"RBF:        K(x,y) = exp(-γ||x-y||²)")
print(f"Sigmoid:    K(x,y) = tanh(γ·xᵀy + r)")

print("\nWHEN TO USE WHICH:")
print("="*60)
print("Linear:  features >> samples (text classification, high-dim)")
print("RBF:     default choice, works for most non-linear problems")
print("Poly:    when you know polynomial relationship exists")

# RBF gamma parameter
print("\nGAMMA (γ) for RBF:")
gammas = [0.01, 0.1, 1.0, 10.0]
for g in gammas:
    svm = SVC(kernel='rbf', C=1.0, gamma=g)
    svm.fit(X_tr_s, y_tr)
    print(f"  γ={g:5.2f}: train={svm.score(X_tr_s, y_tr):.3f}, test={svm.score(X_te_s, y_te):.3f}")

print("\n💡 Large γ → more complex boundary (overfitting risk)")
print("   Small γ → smoother boundary (underfitting risk)")

In [ ]:
# ============================================================
# Hyperparameter Tuning with GridSearchCV
# ============================================================

param_grid = {
    'C': [0.1, 1.0, 10.0],
    'gamma': ['scale', 'auto', 0.1, 1.0],
    'kernel': ['rbf', 'linear']
}

grid = GridSearchCV(SVC(), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_tr_s, y_tr)

print(f"Best params: {grid.best_params_}")
print(f"Best CV acc: {grid.best_score_:.3f}")
print(f"Test acc:    {grid.score(X_te_s, y_te):.3f}")

---

## 🗺️ SVM Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  SVM — Decision Guide                                            ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  KERNEL SELECTION:                                               ║
║    Linear data or high-dim  → kernel='linear'                   ║
║    Non-linear, default      → kernel='rbf'                      ║
║    Known polynomial relation → kernel='poly'                    ║
║                                                                   ║
║  HYPERPARAMETERS:                                                ║
║    C: regularization (high=overfit, low=underfit)               ║
║    γ: RBF influence radius (high=complex, low=smooth)          ║
║                                                                   ║
║  MUST DO:                                                        ║
║    ✓ Always scale features (StandardScaler)                     ║
║    ✓ Use GridSearchCV for C and gamma                           ║
║    ✓ Check n_support_ (too many SVs = problem)                 ║
║                                                                   ║
║  PROS: Effective in high-dim, memory-efficient (only SVs)      ║
║  CONS: Slow for large n, no probabilities (use SVC(prob=True)) ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | What are support vectors? | The closest data points to the decision boundary. Only these determine the boundary |
| 2 | Explain the kernel trick | Maps data to higher dimension without computing the transformation. K(x,y) = φ(x)·φ(y) |
| 3 | C parameter effect? | High C: narrow margin, more complex (overfit). Low C: wide margin, simpler (underfit) |
| 4 | RBF vs Linear kernel? | RBF for non-linear data. Linear for high-dimensional or linearly separable data |
| 5 | Why scale features for SVM? | SVM uses distances. Unscaled features with different ranges bias the distance metric |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • SVM = find max margin hyperplane                     │
## │  • Support vectors = boundary-defining points           │
## │  • Always scale, always tune C and gamma                │
## │  • RBF = default kernel for non-linear                  │
## │  • Linear when features >> samples                      │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Decision Trees & Random Forest** — entropy, Gini, bagging